# Exp1: Historical feature engineering and source ablation

Application data describes the applicant at submission, while historical records reveal debt exposure, past application outcomes and repayment behaviour. This notebook summarises those records into **31 applicant-level features** and tests their incremental predictive value through five feature groups: Application, + Bureau, + Previous, + Repayment and All sources.

The feature set builds on 13 initial aggregates with 18 indicators of delinquency severity, persistence, payment adequacy, application frequency and recent utilisation. Monthly bureau histories complement loan-level debt summaries by capturing the worst recorded delinquency status, the number of delinquent loan-month records and their share. These three indicators belong to **+ Bureau** and **All sources**.

`bb_max_dpd` is an ordinal status bucket, not an exact overdue-day count. `bb_months_with_dpd` counts loan-month records rather than distinct calendar months. Missing status does not trigger a delinquency flag; `bb_ever_late_share` uses all records as its denominator. Applicants with no bureau-balance records retain missing values after joining.

The five groups use the same previously selected gradient-boosting settings, so their comparison focuses on the information contributed by each source. Gender and marital status are excluded; age is retained. The main `exp1.ipynb` separately evaluates the inclusion of both demographic attributes.

**Output folder:** `Exp_1/exp1_outputs/` contains `best_model_scores.csv` (validation/test scores), `sensitivity_scores.csv` (both demographic variants, generated by the main notebook), `exp1_metrics.csv` (model performance), `exp1_comparisons.csv` (paired tests), `features.parquet` (engineered applicant table) and `model_metadata.json` (feature definitions and settings). Intermediate model predictions stay in memory. Existing investigation scores are not overwritten.

**Exported evidence:** all five configurations in this source ablation are also included in the complete evaluation in `evaluate_exp1.ipynb`. They are saved alongside the main model comparison in `exp1_metrics.csv`, with paired source-versus-application tests in `exp1_comparisons.csv`. Application-only and all-sources rows are shared between analyses rather than duplicated. Run the full evaluation notebook to refresh those combined tables.

## 1. Setup and existing features (same as notebook 01)

In [1]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

_shared_candidates = [
    parent / relative
    for parent in (Path.cwd(), *Path.cwd().parents)
    for relative in ("outputs/outputs", "outputs", "hongwei_outputs/outputs")
]
SHARED = next((p.resolve() for p in _shared_candidates
               if (p / "clean_data/splits.csv").is_file()
               and (p / "exp1/eval_exp1.py").is_file()), None)
if SHARED is None:
    raise FileNotFoundError("Cannot find shared clean_data and exp1/eval_exp1.py from this folder.")
CLEAN = SHARED / "clean_data"
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "Exp_1/exp1.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project Exp_1 folder from the current working directory.")
HERE = PROJECT_ROOT / "Exp_1" / "exp1_outputs"          # stable output path from root or Exp_1
HERE.mkdir(exist_ok=True)

splits = pd.read_csv(CLEAN / "splits.csv")
print(splits["split"].value_counts().to_string())

spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)


import json

def write_result_rows(rows, section, record_type, folder=HERE):
    """Intermediate results are displayed only; final CSVs are written in Section 6."""
    return




def update_metadata(values, folder=HERE):
    target = Path(folder) / "model_metadata.json"
    existing = json.loads(target.read_text()) if target.exists() else {}
    existing.update(values)
    target.write_text(json.dumps(existing, indent=2))


def evaluate_and_record(engine, predictions, labels=None, *, stage, folder=HERE, **kwargs):
    metrics, comparisons, evaluated = engine.compare_prediction_files(predictions, labels, **kwargs)
    write_result_rows(metrics, stage, "test_model", folder)
    write_result_rows(comparisons, stage, "paired_comparison", folder)
    return metrics, comparisons, evaluated


In [2]:
EXCLUDE = ["SK_ID_CURR", "TARGET", "split", "CODE_GENDER", "NAME_FAMILY_STATUS"]

app = pd.read_parquet(CLEAN / "application.parquet").drop(columns=["TARGET"])
app = app.merge(splits[["SK_ID_CURR", "TARGET", "split"]], on="SK_ID_CURR", how="inner")
app_cols = [c for c in app.columns if c not in EXCLUDE]
print(f"{len(app):,} applicants, {len(app_cols)} application features")

In [3]:
# Bureau: loans at other lenders
b = pd.read_parquet(CLEAN / "bureau.parquet")
bureau = b.groupby("SK_ID_CURR").agg(
    n_bureau=("SK_ID_BUREAU", "size"),
    bureau_debt=("AMT_CREDIT_SUM_DEBT", "sum"),
    bureau_overdue=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_credit=("AMT_CREDIT_SUM", "sum"),
)
bureau["bureau_debt_ratio"] = bureau["bureau_debt"] / bureau["bureau_credit"].where(bureau["bureau_credit"] > 0)

# Previous Home Credit applications (duplicate submissions removed)
p = pd.read_parquet(CLEAN / "previous_application.parquet")
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["refused"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Refused").astype(int)
prev = p.groupby("SK_ID_CURR").agg(
    n_prev=("SK_ID_PREV", "size"),
    prev_refusal_rate=("refused", "mean"),
    prev_amt_mean=("AMT_APPLICATION", "mean"),
)

# Instalments: one row per instalment (split payments already merged in cleaning)
i = pd.read_parquet(CLEAN / "installments.parquet")
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]) > 0, np.nan)   # unpaid: not counted
i["under"] = (i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)).clip(lower=0)
inst = i.groupby("SK_ID_CURR").agg(
    n_inst=("SK_ID_PREV", "size"),
    late_rate=("late", "mean"),
    underpay_mean=("under", "mean"),
)

# Credit cards: utilisation
c = pd.read_parquet(CLEAN / "credit_card.parquet")
c["util"] = c["AMT_BALANCE"] / c["AMT_CREDIT_LIMIT_ACTUAL"].where(c["AMT_CREDIT_LIMIT_ACTUAL"] > 0)
cc = c.groupby("SK_ID_CURR").agg(cc_util_mean=("util", "mean"), cc_months=("MONTHS_BALANCE", "nunique"))

hist = bureau.join([prev, inst, cc], how="outer").reset_index()
hist_cols = [col for col in hist.columns if col != "SK_ID_CURR"]
print(f"History features: {len(hist_cols)} columns for {len(hist):,} applicants with any history")

In [4]:
feat_old = app.merge(hist, on="SK_ID_CURR", how="left")
BUREAU = ["n_bureau", "bureau_debt", "bureau_overdue", "bureau_credit", "bureau_debt_ratio"]
PREVIOUS = ["n_prev", "prev_refusal_rate", "prev_amt_mean"]
REPAYMENT = ["n_inst", "late_rate", "underpay_mean", "cc_util_mean", "cc_months"]
assert set(BUREAU + PREVIOUS + REPAYMENT) == set(hist_cols)

## 2. Additional indicators of repayment severity, persistence and recent behaviour

In [5]:
# POS / cash loans
q = pd.read_parquet(CLEAN / "pos_cash.parquet", columns=["SK_ID_CURR", "SK_DPD", "CNT_INSTALMENT_FUTURE"])
q["dpd_flag"] = (q["SK_DPD"] > 0).astype("int32")
pos = q.groupby("SK_ID_CURR").agg(
    pos_max_dpd=("SK_DPD", "max"),
    pos_months_dpd=("dpd_flag", "sum"),
    pos_remaining_inst=("CNT_INSTALMENT_FUTURE", "mean"),
)

# Credit card, month by month (one row per applicant-month, summed over contracts)
c = pd.read_parquet(CLEAN / "credit_card.parquet", columns=[
    "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_TOTAL_CURRENT", "SK_DPD"])
c["below_min"] = ((c["AMT_INST_MIN_REGULARITY"] > 0) &
                  (c["AMT_PAYMENT_TOTAL_CURRENT"].fillna(0) < c["AMT_INST_MIN_REGULARITY"])).astype("int32")
c["dpd_flag"] = (c["SK_DPD"] > 0).astype("int32")
c["min_ratio"] = c["AMT_PAYMENT_TOTAL_CURRENT"] / c["AMT_INST_MIN_REGULARITY"].where(c["AMT_INST_MIN_REGULARITY"] > 0)
m = c.groupby(["SK_ID_CURR", "MONTHS_BALANCE"], as_index=False).agg(
    bal=("AMT_BALANCE", "sum"), lim=("AMT_CREDIT_LIMIT_ACTUAL", "sum"),
    below=("below_min", "max"), dpd=("dpd_flag", "max"), min_ratio=("min_ratio", "mean"))
m["util"] = m["bal"] / m["lim"].where(m["lim"] > 0)
recent = m[m["MONTHS_BALANCE"] >= -3].groupby("SK_ID_CURR")["util"].mean()
card = m.groupby("SK_ID_CURR").agg(
    cc_months_below_min=("below", "sum"),
    cc_months_dpd=("dpd", "sum"),
    cc_min_ratio_mean=("min_ratio", "mean"),
    cc_util_overall=("util", "mean"),
)
card["cc_balance_trend"] = recent.reindex(card.index) - card["cc_util_overall"]
card = card.drop(columns="cc_util_overall")

# Instalments
i = pd.read_parquet(CLEAN / "installments.parquet", columns=[
    "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"])
paid = i["DAYS_ENTRY_PAYMENT"].notna()
i["days_late"] = np.where(paid, (i["DAYS_ENTRY_PAYMENT"] - i["DAYS_INSTALMENT"]).clip(lower=0), np.nan)
i["under_flag"] = ((i["AMT_INSTALMENT"] - i["AMT_PAYMENT"].fillna(0)) > 1.0).astype("int32")
i["paid_amt"] = i["AMT_PAYMENT"].fillna(0)
inst_new = i.groupby("SK_ID_CURR").agg(
    inst_max_days_late=("days_late", "max"),
    inst_mean_days_late=("days_late", "mean"),
    inst_underpay_freq=("under_flag", "mean"),
    paid_sum=("paid_amt", "sum"),
    due_sum=("AMT_INSTALMENT", "sum"),
)
inst_new["inst_paid_to_due"] = inst_new["paid_sum"] / inst_new["due_sum"].where(inst_new["due_sum"] > 0)
inst_new = inst_new.drop(columns=["paid_sum", "due_sum"])

# Bureau (loan-level)
b = pd.read_parquet(CLEAN / "bureau.parquet", columns=["SK_ID_CURR", "CREDIT_DAY_OVERDUE", "CREDIT_ACTIVE"])
bureau_new = b.groupby("SK_ID_CURR").agg(
    bureau_max_days_overdue=("CREDIT_DAY_OVERDUE", "max"),
    bureau_active_count=("CREDIT_ACTIVE", lambda s: (s.astype(str) == "Active").sum()),
)

# Bureau-balance summaries of monthly delinquency at external lenders
bb_flat = pd.read_parquet(CLEAN / "bureau_balance.parquet", columns=["SK_ID_CURR", "STATUS_DPD"])
bb_flat["dpd_flag"] = (bb_flat["STATUS_DPD"].fillna(0) > 0).astype("int32")
bb_agg = bb_flat.groupby("SK_ID_CURR").agg(
    bb_max_dpd=("STATUS_DPD", "max"),
    bb_months_with_dpd=("dpd_flag", "sum"),
    bb_ever_late_share=("dpd_flag", "mean"),
)
print(f"Bureau balance flat: {len(bb_agg):,} applicants")

# Previous applications
p = pd.read_parquet(CLEAN / "previous_application.parquet", columns=[
    "SK_ID_CURR", "NAME_CONTRACT_STATUS", "DAYS_DECISION", "IS_DUPLICATE_SUBMISSION"])
p = p[p["IS_DUPLICATE_SUBMISSION"] == 0].copy()
p["approved"] = (p["NAME_CONTRACT_STATUS"].astype(str) == "Approved").astype("int32")
p["last_year"] = (p["DAYS_DECISION"] >= -365).astype("int32")
prev_new = p.groupby("SK_ID_CURR").agg(prev_approval_rate=("approved", "mean"), prev_apps_last_year=("last_year", "sum"))

new_hist = pos.join([card, inst_new, bureau_new, bb_agg, prev_new], how="outer").reset_index()
NEW_POS = ["pos_max_dpd", "pos_months_dpd", "pos_remaining_inst"]
NEW_CARD = ["cc_months_below_min", "cc_months_dpd", "cc_min_ratio_mean", "cc_balance_trend"]
NEW_INST = ["inst_max_days_late", "inst_mean_days_late", "inst_underpay_freq", "inst_paid_to_due"]
NEW_BUREAU = ["bureau_max_days_overdue", "bureau_active_count"]
NEW_BB = ["bb_max_dpd", "bb_months_with_dpd", "bb_ever_late_share"]
NEW_PREV = ["prev_approval_rate", "prev_apps_last_year"]
assert set(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV) == set(new_hist.columns) - {"SK_ID_CURR"}
print(f"New features: {len(new_hist.columns) - 1} columns for {len(new_hist):,} applicants")
display(new_hist.describe().T[["count", "mean", "min", "max"]].round(3))

,count,mean,min,max
SK_ID_CURR,99290.0,277928.762,100011.00,456248.000
pos_max_dpd,94164.0,15.262,0.00,4231.000
pos_months_dpd,94164.0,0.885,0.00,120.000
pos_remaining_inst,94155.0,9.215,0.00,59.500
cc_months_below_min,28368.0,2.096,0.00,70.000
cc_months_dpd,28368.0,1.546,0.00,94.000
cc_min_ratio_mean,19351.0,34.067,0.00,471288.531
cc_balance_trend,21320.0,0.007,-0.97,2.767
inst_max_days_late,94873.0,18.639,0.00,2852.000
inst_mean_days_late,94873.0,1.282,0.00,2813.000


## 3. Join all features and define the groups

In [6]:
feat = feat_old.merge(new_hist, on="SK_ID_CURR", how="left")
assert feat["SK_ID_CURR"].is_unique

GROUPS = {
    "Application": app_cols,
    "+ Bureau": app_cols + BUREAU + NEW_BUREAU + NEW_BB,
    "+ Previous": app_cols + PREVIOUS + NEW_PREV,
    "+ Repayment": app_cols + REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
    "All sources": app_cols + hist_cols + NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV,
}
# 31 historical features spanning loan summaries and monthly repayment behaviour.
assert len(hist_cols) == 13
assert len(NEW_POS + NEW_CARD + NEW_INST + NEW_BUREAU + NEW_BB + NEW_PREV) == 18
assert len(GROUPS["All sources"]) == len(app_cols) + 31
assert len(GROUPS["All sources"]) == len(set(GROUPS["All sources"]))


## 4. Fit each group on train (same settings as notebook 03)

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

PARAMS = dict(learning_rate=0.03, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=1.0)
FIXED = dict(max_iter=1000, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)

def model_inputs(cols):
    return pd.get_dummies(feat[cols], dummy_na=True).astype("float32")

FILES = {}
for name, cols in GROUPS.items():
    X = model_inputs(cols)
    tr = feat["split"] == "train"
    sc = feat["split"].isin(["val", "test"])
    model = HistGradientBoostingClassifier(**PARAMS, **FIXED).fit(X[tr], feat.loc[tr, "TARGET"])
    out = feat.loc[sc, ["SK_ID_CURR"]].assign(score=model.predict_proba(X[sc])[:, 1])
    FILES[name] = out
    print(f"{name:<12} inputs={X.shape[1]:>4}  trees={model.n_iter_:>4} (predictions kept in memory)")
rich_predictions = {name: df.copy() for name, df in FILES.items()}


## 5. Evaluate with the shared code (`eval_exp1.py`), 1000 bootstrap resamples

The table in this feature notebook adjusts across its eight source-ablation tests. The complete `evaluate_exp1.ipynb` adjusts across all sixteen planned tests, so its adjusted p-values can be larger. Use the refreshed `exp1_comparisons.csv` when reporting the combined experiment.


In [8]:
spec = importlib.util.spec_from_file_location("eval_exp1", SHARED / "exp1" / "eval_exp1.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

labels = list(GROUPS)
table, comps, _ = evaluate_and_record(ev, [FILES[l] for l in labels], labels, n_boot=1000, stage="ablation_31_history")
display(ev.pretty_metrics(table))
display(ev.pretty_comparisons(comps))

,threshold,pr_auc,roc_auc,precision,recall,f1
model,,,,,,
Application,0.166,0.215 (0.196-0.237),0.741 (0.726-0.755),0.235 (0.217-0.254),0.350 (0.324-0.376),0.281 (0.260-0.301)
+ Bureau,0.157,0.223 (0.204-0.245),0.743 (0.728-0.757),0.229 (0.212-0.246),0.381 (0.355-0.406),0.286 (0.267-0.305)
+ Previous,0.150,0.222 (0.202-0.243),0.743 (0.728-0.757),0.221 (0.204-0.238),0.391 (0.364-0.419),0.282 (0.264-0.302)
+ Repayment,0.145,0.242 (0.221-0.265),0.757 (0.742-0.771),0.230 (0.213-0.247),0.433 (0.406-0.459),0.300 (0.281-0.319)
All sources,0.144,0.250 (0.228-0.273),0.758 (0.743-0.772),0.230 (0.214-0.248),0.438 (0.410-0.467),0.302 (0.282-0.321)


,model,vs_baseline,metric,difference (95% CI),p_value,p_holm,verdict
0,+ Bureau,Application,pr_auc,+0.0080 (+0.0013 to +0.0139),0.026,0.1,no significant difference
1,+ Bureau,Application,roc_auc,+0.0021 (-0.0011 to +0.0052),0.23,0.45,no significant difference
2,+ Previous,Application,pr_auc,+0.0067 (+0.0005 to +0.0125),0.038,0.11,no significant difference
3,+ Previous,Application,roc_auc,+0.0019 (-0.0010 to +0.0048),0.23,0.45,no significant difference
4,+ Repayment,Application,pr_auc,+0.0271 (+0.0170 to +0.0361),0,0,significantly better
5,+ Repayment,Application,roc_auc,+0.0155 (+0.0104 to +0.0211),9e-09,5.4e-08,significantly better
6,All sources,Application,pr_auc,+0.0351 (+0.0233 to +0.0455),0,0,significantly better
7,All sources,Application,roc_auc,+0.0165 (+0.0106 to +0.0225),1.7e-07,8.7e-07,significantly better


## 6. Save the applicant-level feature table (for teammates)

In [9]:
feat.drop(columns=["split", "TARGET"]).to_parquet(HERE / "features.parquet", index=False)
history_manifest = {
    "bureau": BUREAU + NEW_BUREAU + NEW_BB,
    "previous": PREVIOUS + NEW_PREV,
    "repayment": REPAYMENT + NEW_POS + NEW_CARD + NEW_INST,
}
assert sum(map(len, history_manifest.values())) == 31
update_metadata({"history_features": history_manifest})
rich_predictions["All sources"].to_csv(HERE / "best_model_scores.csv", index=False)
print("Saved features.parquet, best_model_scores.csv and feature definitions in model_metadata.json")
